# PromptGFM-Bio — Google Colab Training Notebook
**Gene-Phenotype Prediction for Rare Diseases**

### ✅ Fully Automatic Session Persistence via Google Drive
Everything is handled for you automatically:

| Asset | Saved to Drive | Restored next session |
|---|---|---|
| BioBERT weights (~440 MB) | After download | Auto-detected & skipped |
| Processed graph (~600 MB) | Right after building | Auto-detected & skipped |
| Checkpoints | **After every epoch + end of session** | Auto-detected & resumed |

**You never need to set any flags or run save cells manually.**

> ⚙️ **Recommended runtime:** Runtime → Change runtime type → **T4 GPU** (free) or **A100 GPU** (Colab Pro+)

## 0. Mount Google Drive
Must be the first cell you run — everything else depends on it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

## 1. Environment Check

In [ ]:
import sys, subprocess, os
import torch

print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

## 2. ⚙️ Configuration & Auto-Detection
**Nothing to edit** — this cell auto-detects what exists in Drive and sets all flags for you.

Only change `DRIVE_ROOT` if you want assets stored in a different Drive folder.

In [ ]:
import os as _os
from pathlib import Path as _Path

# ─── ONLY USER-EDITABLE SETTING ──────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/promptgfm'

# ─── DRIVE PATHS ─────────────────────────────────────────────────────────
DRIVE_HF_CACHE    = f'{DRIVE_ROOT}/promptgfm-model-cache'
DRIVE_DATA        = f'{DRIVE_ROOT}/promptgfm-data'
DRIVE_CHECKPOINTS = f'{DRIVE_ROOT}/promptgfm-checkpoints'

# ─── LOCAL PATHS (fast I/O during training) ───────────────────────────────
LOCAL_PROJECT     = '/content/PromptGFM-Bio'
HF_CACHE_DIR      = '/content/hf_cache'

_os.environ['HF_HOME']               = HF_CACHE_DIR
_os.environ['TRANSFORMERS_CACHE']    = HF_CACHE_DIR
_os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR

# ─── AUTO-DETECT RESUME FLAGS ─────────────────────────────────────────────
# Check Drive for each asset and set flags automatically.
def _has_hf_cache():
    return (_Path(DRIVE_HF_CACHE) / 'hf_cache.tar.gz').exists()

def _has_graph_data():
    return (_Path(DRIVE_DATA) / 'data.tar.gz').exists()

def _has_checkpoints():
    p = _Path(DRIVE_CHECKPOINTS)
    return p.exists() and any(p.glob('checkpoint_epoch_*.pt'))

RESUME_HF_CACHE    = _has_hf_cache()
RESUME_DATA        = _has_graph_data()
RESUME_CHECKPOINTS = _has_checkpoints()

# Create Drive folders so they're ready for saves later
for _d in [DRIVE_HF_CACHE, DRIVE_DATA, DRIVE_CHECKPOINTS]:
    _Path(_d).mkdir(parents=True, exist_ok=True)

print('── Auto-detected configuration ──────────────────────────────')
print(f'  RESUME_HF_CACHE    = {RESUME_HF_CACHE}  {"(will restore BioBERT from Drive)" if RESUME_HF_CACHE else "(will download fresh)"}')
print(f'  RESUME_DATA        = {RESUME_DATA}  {"(will restore graph from Drive)" if RESUME_DATA else "(will download + preprocess fresh)"}')
print(f'  RESUME_CHECKPOINTS = {RESUME_CHECKPOINTS}  {"(will resume training)" if RESUME_CHECKPOINTS else "(will train from scratch)"}')
print()
print(f'  Drive root     : {DRIVE_ROOT}')
print(f'  HF cache (local): {HF_CACHE_DIR}')
print()
if RESUME_HF_CACHE:
    print('  BioBERT  ✅ found in Drive')
else:
    print('  BioBERT  ○ not in Drive — will download (~440 MB, ~3 min)')
if RESUME_DATA:
    print('  Graph    ✅ found in Drive')
else:
    print('  Graph    ○ not in Drive — will build (~25 min first time)')
if RESUME_CHECKPOINTS:
    _ckpt_files = sorted(_Path(DRIVE_CHECKPOINTS).glob('checkpoint_epoch_*.pt'))
    _epochs = [f.stem.replace('checkpoint_epoch_', '') for f in _ckpt_files]
    print(f'  Checkpoints ✅ found in Drive — epochs: {_epochs}')
else:
    print('  Checkpoints ○ not in Drive — training from epoch 0')

## 3. Install PyTorch Geometric

In [ ]:
import torch, subprocess, sys

TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = torch.version.cuda.replace('.', '')
WHEEL_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html'
print(f'PyG wheel source: {WHEEL_URL}')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '-f', WHEEL_URL,
     'torch-scatter', 'torch-sparse', 'torch-cluster',
     'torch-spline-conv', 'torch-geometric'],
    check=True
)
print('✅ PyTorch Geometric installed')

## 4. Install Extra Dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade', 'setuptools', 'wheel', 'pip'], check=True)

extra = [
    'transformers>=4.40.0',
    'sentence-transformers>=2.7.0',
    'biopython>=1.84',
    'networkx>=3.2',
    'wandb>=0.17.0',
    'python-dotenv>=1.0.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + extra, check=True)
print('✅ Extra packages installed')

## 5. Clone Project Code from GitHub

In [ ]:
import os, subprocess, sys
from pathlib import Path

os.environ['GIT_TERMINAL_PROMPT'] = '0'

GITHUB_URL  = 'https://github.com/pes1ug23am910/PromptGFM-Bio.git'
PROJECT_DIR = LOCAL_PROJECT

_github_token = ''
try:
    from google.colab import userdata
    _github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

_auth_url = GITHUB_URL.replace('https://', f'https://{_github_token}@') if _github_token else GITHUB_URL

if not os.path.exists(PROJECT_DIR):
    result = subprocess.run(['git', 'clone', '--depth', '1', _auth_url, PROJECT_DIR])
    if result.returncode != 0:
        raise RuntimeError(
            'Git clone failed.\n'
            '  • Private repo: add token via Tools → Secrets → GITHUB_TOKEN\n'
            f'  • Public repo: check URL is correct: {GITHUB_URL}'
        )
    print(f'✅ Cloned to {PROJECT_DIR}')
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'])
    print('✅ Pulled latest changes')

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

## 6. Create Directory Structure

In [ ]:
from pathlib import Path

dirs = [
    'data/raw', 'data/processed', 'data/splits',
    'checkpoints/promptgfm_film',
    'logs',
]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)
print('✅ Directories created')

## 7. 🔄 Drive Sync Helper
Defines the `DriveSync` background thread used throughout the notebook to automatically sync checkpoints to Drive as they are written — **no manual action needed**.

> This cell only defines the helper. The thread is started in the training cell.

In [ ]:
import shutil, threading, time, tarfile
from pathlib import Path

class DriveSync:
    """
    Background thread that watches a local directory and automatically
    copies new or updated files to a Google Drive destination folder.

    Usage:
        syncer = DriveSync(src='checkpoints/promptgfm_film', dst=DRIVE_CHECKPOINTS)
        syncer.start()
        # ... run training (blocking) ...
        syncer.stop()           # stops the watcher loop
        syncer.sync_now()       # final full sync before exiting
    """

    def __init__(self, src: str, dst: str, poll_seconds: float = 15.0):
        self.src = Path(src)
        self.dst = Path(dst)
        self.poll = poll_seconds
        self._stop_event = threading.Event()
        self._seen: dict = {}          # filename → mtime of last sync
        self._thread = threading.Thread(target=self._run, daemon=True)
        self.dst.mkdir(parents=True, exist_ok=True)

    def start(self):
        self._stop_event.clear()
        self._thread.start()
        print(f'🔄 DriveSync started  {self.src} → {self.dst}  (every {self.poll}s)')

    def stop(self):
        self._stop_event.set()

    def _run(self):
        while not self._stop_event.is_set():
            self._sync_once()
            self._stop_event.wait(self.poll)

    def _sync_once(self):
        if not self.src.exists():
            return
        for f in self.src.glob('*.pt'):
            mtime = f.stat().st_mtime
            if self._seen.get(f.name) != mtime:
                dst_file = self.dst / f.name
                shutil.copy2(f, dst_file)
                self._seen[f.name] = mtime
                print(f'  💾 Synced → Drive: {f.name}')

    def sync_now(self):
        """Force a full sync immediately (call after training finishes)."""
        print('🔄 Final checkpoint sync to Drive...')
        self._sync_once()
        # Also copy best_model.pt and any other .pt files
        if self.src.exists():
            for f in self.src.glob('*.pt'):
                shutil.copy2(f, self.dst / f.name)
                print(f'  💾 Synced → Drive: {f.name}')
        print('  ✅ Checkpoint sync complete')

print('✅ DriveSync helper defined')

## 8. Restore Assets from Drive
Automatically restores whatever was previously saved — no flags to set.

**First-time run**: all three blocks print "not found" and assets are built fresh, then auto-saved to Drive for you.

In [ ]:
import shutil, tarfile
from pathlib import Path

def restore_tar(src_path, dest_dir, label):
    src = Path(src_path)
    print(f'  checking : {src}  (exists={src.exists()})')
    if src.exists():
        dest = Path(dest_dir)
        dest.mkdir(parents=True, exist_ok=True)
        with tarfile.open(src, 'r:gz') as tar:
            tar.extractall(dest)
        print(f'  ✅ {label} restored')
        return True
    return False

def restore_dir(src_path, dest_dir, label):
    src = Path(src_path)
    print(f'  checking : {src}  (exists={src.exists()})')
    if src.exists() and any(src.iterdir()):
        dest = Path(dest_dir)
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src, dest)
        print(f'  ✅ {label} restored')
        return True
    return False

# ── A. HuggingFace model cache ────────────────────────────────────────────
print('[A] HF model cache')
if RESUME_HF_CACHE:
    ok = restore_tar(f'{DRIVE_HF_CACHE}/hf_cache.tar.gz', HF_CACHE_DIR, 'HF model cache')
    if not ok:
        ok = restore_dir(f'{DRIVE_HF_CACHE}/hf_cache', HF_CACHE_DIR, 'HF model cache')
    if not ok:
        print('  ⚠️  Not found on Drive — BioBERT will be re-downloaded and re-saved')
        RESUME_HF_CACHE = False
else:
    print('  ○ Not on Drive — will download fresh')

# ── B. Preprocessed graph + splits ───────────────────────────────────────
print()
print('[B] Graph data')
if RESUME_DATA:
    ok = restore_tar(f'{DRIVE_DATA}/data.tar.gz', 'data', 'Graph data')
    if not ok:
        ok = restore_dir(f'{DRIVE_DATA}/processed', 'data/processed', 'Processed graph')
        restore_dir(f'{DRIVE_DATA}/splits', 'data/splits', 'Data splits')
    graph = Path('data/processed/biomedical_graph.pt')
    if graph.exists():
        print(f'  ✅ Graph ready ({graph.stat().st_size/1e6:.0f} MB)')
    else:
        print('  ⚠️  Graph missing — will preprocess fresh and auto-save')
        RESUME_DATA = False
else:
    print('  ○ Not on Drive — will download + preprocess fresh')

# ── C. Training checkpoints ───────────────────────────────────────────────
print()
print('[C] Checkpoints')
if RESUME_CHECKPOINTS:
    ckpt_src = Path(DRIVE_CHECKPOINTS)
    ckpt_dst = Path('checkpoints/promptgfm_film')
    ckpt_dst.mkdir(parents=True, exist_ok=True)
    files_copied = 0
    for f in ckpt_src.rglob('*.pt'):
        shutil.copy2(f, ckpt_dst / f.name)
        files_copied += 1
    if files_copied:
        epochs = sorted([f.stem.replace('checkpoint_epoch_', '')
                         for f in ckpt_dst.glob('checkpoint_epoch_*.pt')])
        print(f'  ✅ {files_copied} checkpoint(s) restored. Epochs: {epochs}')
    else:
        print('  ⚠️  No .pt files found on Drive — training from scratch')
        RESUME_CHECKPOINTS = False
else:
    print('  ○ Not on Drive — will train from scratch')

## 9. Download & Cache BioBERT
Downloads PubMedBERT weights (~440 MB) to local disk, then **automatically backs them up to Drive**.

Next session this step is skipped entirely — weights are restored from Drive in Step 8.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import warnings, os, tarfile, shutil

MODEL_NAME = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext'

hub_dir = Path(HF_CACHE_DIR) / 'hub'
model_cache_name = 'models--' + MODEL_NAME.replace('/', '--')
model_snapshots = hub_dir / model_cache_name / 'snapshots'

if RESUME_HF_CACHE and model_snapshots.exists() and any(model_snapshots.iterdir()):
    print('⏭️  BioBERT already restored from Drive — skipping download')
else:
    print(f'Downloading {MODEL_NAME}')
    print(f'  Size : ~440 MB — takes ~2-3 min')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)
        snapshot_download(
            repo_id=MODEL_NAME,
            cache_dir=HF_CACHE_DIR,
            ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*', 'rust_model*', '*.ot'],
        )
    print('\n✅ BioBERT downloaded')

    # ── Auto-save to Drive ──────────────────────────────────────────────
    print('\n💾 Saving BioBERT cache to Drive (one-time, ~2 min)...')
    _out = Path(DRIVE_HF_CACHE) / 'hf_cache.tar.gz'
    with tarfile.open(_out, 'w:gz') as tar:
        tar.add(HF_CACHE_DIR, arcname='hf_cache')
    print(f'  ✅ Saved to {_out}  ({_out.stat().st_size/1e6:.0f} MB)')
    print('  Next session: BioBERT will be restored from Drive automatically.')

os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE']       = '1'
print('✅ Offline mode enabled — training loads BioBERT from disk only')

## 10. Download Biomedical Datasets
Skipped automatically if graph was restored from Drive in Step 8.

In [ ]:
from pathlib import Path

graph_exists = Path('data/processed/biomedical_graph.pt').exists()

if RESUME_DATA and graph_exists:
    print('⏭️  Download skipped — graph already restored from Drive')
else:
    print('Downloading datasets (~1.5 GB, ~10 min)...')
    result = subprocess.run(
        [sys.executable, 'scripts/download_data.py', '--dataset', 'all'],
        capture_output=False
    )
    print('Download exit code:', result.returncode)

## 11. Preprocess Data (Build Knowledge Graph)
Skipped automatically if graph was restored from Drive in Step 8.

In [ ]:
from pathlib import Path

graph_path = Path('data/processed/biomedical_graph.pt')

if RESUME_DATA and graph_path.exists():
    print(f'⏭️  Preprocessing skipped — graph ready ({graph_path.stat().st_size/1e6:.0f} MB)')
else:
    print('Building knowledge graph (~25 min first time)...')
    result = subprocess.run(
        [sys.executable, 'scripts/preprocess_all.py'],
        capture_output=False
    )
    print('Preprocessing exit code:', result.returncode)
    if graph_path.exists():
        print(f'✅ Graph ready ({graph_path.stat().st_size/1e6:.0f} MB)')
    else:
        raise RuntimeError('Graph file not created — check logs above')

## 11.5. 💾 Auto-Save Graph to Drive
Saves the processed graph to Drive immediately after building — **runs automatically, no action needed**.

If training crashes after this point, next session will skip straight past download and preprocessing.

In [ ]:
import tarfile
from pathlib import Path

graph_path = Path('data/processed/biomedical_graph.pt')

if RESUME_DATA:
    print('⏭️  Graph was restored from Drive — no re-save needed')
elif not graph_path.exists():
    print('⚠️  Graph not found locally — skipping Drive save')
else:
    print('💾 Saving graph data to Drive...')
    _out = Path(DRIVE_DATA) / 'data.tar.gz'
    _saved = []
    with tarfile.open(_out, 'w:gz') as _tar:
        for _d in ['data/processed', 'data/splits']:
            _p = Path(_d)
            if _p.exists() and any(_p.rglob('*')):
                _tar.add(_p, arcname=_p.name)
                _saved.append(_p.name)
    if _saved:
        print(f'  ✅ Saved {", ".join(_saved)} → {_out}  ({_out.stat().st_size/1e6:.0f} MB)')
        print('  Next session: preprocessing will be skipped automatically.')
    else:
        print('  ⚠️  Nothing to save — data/processed and data/splits are empty')

## 12. W&B Login (Optional)
Add your key via Tools → Secrets → `WANDB_API_KEY`. If absent, W&B is silently disabled.

In [ ]:
WANDB_API_KEY = ''
try:
    from google.colab import userdata
    WANDB_API_KEY = userdata.get('WANDB_API_KEY')
except Exception:
    pass

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print('✅ W&B logged in')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled — metrics logged to stdout only')

## 13. Train
**Checkpoints are automatically synced to Drive after every epoch** via a background thread.
If the session disconnects mid-training, the next session resumes from the last Drive-synced epoch — automatically.

- 🔄 DriveSync thread polls every 15 s for new `.pt` files and copies them to Drive
- 💾 Final full sync runs after training completes
- 🚀 Multi-GPU (torchrun) used automatically when 2+ GPUs are detected

In [ ]:
import os
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE']       = '1'
print('✅ Offline mode confirmed')

In [ ]:
import subprocess, os
os.environ['GIT_TERMINAL_PROMPT'] = '0'
result = subprocess.run(['git', '-C', LOCAL_PROJECT, 'pull'])
if result.returncode == 0:
    print('✅ Latest code pulled from GitHub')
else:
    print('⚠️  git pull skipped (OK if repo is private or offline)')

In [ ]:
import torch as _torch
from pathlib import Path as _Path

# ── Detect in-session checkpoints (same runtime, stopped+restarted) ──────
_ckpt_dir = _Path('checkpoints/promptgfm_film')
_ckpts = sorted(
    _ckpt_dir.glob('checkpoint_epoch_*.pt'),
    key=lambda p: int(p.stem.split('_')[-1])
) if _ckpt_dir.exists() else []

RESUME = RESUME_CHECKPOINTS or bool(_ckpts)

base_args = ['scripts/train.py', '--config', 'configs/kaggle_config.yaml']
if RESUME and _ckpts:
    _last_ckpt = str(_ckpts[-1])
    base_args += ['--resume-checkpoint', _last_ckpt]
    print(f'Resuming from: {_last_ckpt}')
elif RESUME:
    print('RESUME_CHECKPOINTS=True but no local checkpoints — starting fresh')
    RESUME = False

# ── Start background Drive sync BEFORE launching training ────────────────
_syncer = DriveSync(
    src='checkpoints/promptgfm_film',
    dst=DRIVE_CHECKPOINTS,
    poll_seconds=15.0
)
_syncer.start()

# ── Launch training ───────────────────────────────────────────────────────
n_gpus = _torch.cuda.device_count()
print(f'GPUs available: {n_gpus}')

if n_gpus > 1:
    import shutil as _sh
    _torchrun = _sh.which('torchrun') or 'torchrun'
    cmd = [_torchrun, f'--nproc_per_node={n_gpus}', '--master_port=29500'] + base_args
    print(f'Launching DDP on {n_gpus} GPUs')
else:
    cmd = [sys.executable] + base_args
    print('Single-GPU launch')

print('Running:', ' '.join(cmd))
_result = subprocess.run(cmd, capture_output=False)
print('Training exit code:', _result.returncode)

# ── Stop watcher, then final full sync ───────────────────────────────────
_syncer.stop()
_syncer.sync_now()
print()
print('✅ Training complete. All checkpoints are on Drive.')
print(f'   Location: {DRIVE_CHECKPOINTS}')

## 14. Quick Evaluation

In [ ]:
from pathlib import Path

best = Path('checkpoints/promptgfm_film/best_model.pt')
if not best.exists():
    print('No best_model.pt yet — run more training epochs first')
else:
    result = subprocess.run(
        [sys.executable, 'scripts/evaluate.py',
         '--config', 'configs/kaggle_config.yaml',
         '--checkpoint', str(best)],
        capture_output=False
    )
    print('Evaluation exit code:', result.returncode)